<font size=6>In-class Introduction to Curve Fitting

Here, we will begin to use optimization and curve fitting to extract physical parameters from data. We will use a few types of data sets, including a stretched exponential decay and a peak fitting function. Next week, we will start with some X-ray models - I was not able to get the scientific material to catch up to the analysis material.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

!git clone https://github.com/cbishop4/MSE7530.git

<font size=4> **Scipy.optimize**  
SciPy optimize provides functions for minimizing (or maximizing) objective functions, possibly subject to constraints. It includes solvers for nonlinear problems (with support for both local and global optimization algorithms), linear programming, constrained and nonlinear least-squares, root finding, and curve fitting. https://docs.scipy.org/doc/scipy/reference/optimize.html

You should understand the math from lecture for in-class exams, but you don't really need to know it for the exercises today.

We will start with scipy.optimize.curve_fit, which uses non-linear least squares to fit a function f to data.

In [ ]:
from scipy.optimize import curve_fit

## Stretched Exponential  
A stretched exponential function is often used as a phenomenological description of relaxation in disordered systems. Later in the semester, we will return to X-rays, specifically X-ray photon correlation spectroscopy.  

Start with a user-defined function. A stretched exponential function is represented by  
<font size=5>$f(t) = Ae^{-(\frac{t}{\tau_{KWW}})^{\beta}} + C$  
</font>Where $f(t)$ corresponds to some measured property at time $t$,  
$t$ is the time past $t_0$,  
$\tau_{KWW}$ is a characteristic relaxation time that is usually what we are solving for,  
$\beta$ is a stretching exponent that is a measure of how *heterogeneous* the dynamics are (and can also be somewhat nebulous),  
$A$ is the value of the property at $t_0$,  
and $C$ is the value of the property at infinite $t$.

In [ ]:
def KWW(time, A, tau,beta,C):
  ''' A stretched exponential function.
    Args:
      time (float) : time past t_0
      A (float) : value of property at t=0
      tau (float) : characteristic relaxation time
      beta (float) : stretching exponent
      C (float) : value of property at infinite time
    Returns:
      A * np.exp(-(time/tau)**beta) + C (float) : value of property at time
  '''
  return A * np.exp(-(time/tau)**beta) + C

params_in_order = ['A', 'tau', 'beta', 'C'] # will help us later

### Creating a dictionary
We will use a dictionary, a built-in Python data type, to hold the data for how the structure evolves in Polylactic Acid Glasses. Data is from T. Bennin et. al., "Enhanced Segmental Dynamics of Poly(lactic acid) Glasses during Constant Strain Rate Deformation", *Macromolecules* 52 (17): 6428-6437 (2019). https://doi.org/10.1021/acs.macromol.9b01363

In [ ]:
temps = [315, 317, 320, 323]
decays = {}
for T in temps:
  data = pd.read_csv(f'/content/MSE7530/sampledata/{T}K.csv', header=None, names=['t','I'])
  decays[T] = data

We have created a dictionary that holds 4 dataframes as **values**; there is a separate **key** that corresponds to each value. In this case, our keys are integers. They can be any data type, and are often strings.

In [ ]:
print(f'The keys for our dictionary are {decays.keys()}')

You can use a key to access a single dataframe for the temperature that you want.

In [ ]:
decays[315]

### Plotting the data

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,4))
for k in decays.keys():
  for a in ax:
    a.plot(decays[k]['t'],decays[k]['I'],'o',label=f'{k}K')
for a in ax:
  a.set_xlabel('Time (s)')
  a.set_ylabel('Intensity')
  a.legend()
ax[1].semilogx()
ax[0].set_title('Linear Scale')
ax[1].set_title('Log X-Scale')


### Fitting the data

#### 320 K

In [ ]:
result = curve_fit(KWW, decays[320]['t'], decays[320]['I'])

In [ ]:
result

In [ ]:
type(result)

In [ ]:
len(result)

Usually, people will separate the result so that it gives two different things (this is identical to above, just different notation)

In [ ]:
popt, pcov = curve_fit(KWW, decays[320]['t'], decays[320]['I'])

pcov is the covariance matrix. Its diagonals provide the variance of the parameter estimate and allow you to compute standard deviation errors; we will do this later. For now, let's focus on popt, which is the optimal parameters.

In [ ]:
print(f'The optimal parameters are {popt}, corresponding to {params_in_order}')

In [ ]:
fig, ax = plt.subplots()
ax.plot(decays[320]['t'],decays[320]['I'],'o',label='Data')
ax.plot(decays[320]['t'],KWW(decays[320]['t'],*popt),label='Fit')
ax.legend()
ax.set_xlabel('Time (s)')
ax.set_ylabel('Intensity')

#### 323 K

<font color='blue'><font size=4> Repeat this for the T = 323 K data.

In [ ]:
# your code here (add extra cells as needed)

What happened? We need to provide bounds for our parameters and/or fix them based on physical intuition. One way we can fix parameters is by re-defining the equation:

### Constraining from the function

In [ ]:
def KWW2(time, tau):
  ''' A stretched exponential function.
    Args:
      time (float) : time past t_0
      A (float) : value of property at t=0
      tau (float) : characteristic relaxation time
      beta (float) : stretching exponent
      C (float) : value of property at infinite time
    Returns:
      A * np.exp(-(time/tau)**beta) + C (float) : value of property at time
  '''
  beta = 0.5
  C = 0.0
  A = 1.0
  return A * np.exp(-(time/tau)**beta) + C

In [ ]:
popt, pcov = curve_fit(KWW2, decays[323]['t'], decays[323]['I'])

pcov is the covariance matrix. Its diagonals provide the variance of the parameter estimate and allow you to compute standard deviation errors; we will do this later. For now, let's focus on popt, which is the optimal parameters.

In [ ]:
print(f'The optimal parameters is {popt}, which was our only fit parameter')

In [ ]:
fig, ax = plt.subplots()
ax.plot(decays[323]['t'],decays[323]['I'],'o',label='Data')
ax.plot(decays[323]['t'],KWW2(decays[323]['t'],*popt),label='Fit')
ax.legend()
ax.set_xlabel('Time (s)')
ax.set_ylabel('Intensity')

#### Solving for all 4 $\tau$

<font color=blue>Run through this for all 4 using the dictionary. We will do this together. Finish with a plot of $\tau$ vs. Temperature.